# Teen Mental Health & Social Media Behavior Analysis
**Objective:** To clean, transform, and analyze how screen habits, platform choices, and sleep affect stress, anxiety, and academic performance among teenagers.

### Tech Stack used:
* **Pandas:** For structural data manipulation, aggregations, and grouping.
* **NumPy:** For high-performance array operations, multi-condition masking, and statistical evaluation.

In [1]:
import pandas as pd
import numpy as np

# 1. Load the dataset
df = pd.read_csv('Teen_Mental_Health_Dataset.csv')

# 2. Inspect basic metadata
print("--- Dataset Shape ---")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")

print("\n--- Data Column Types ---")
print(df.info())

print("\n--- Missing Value Audit ---")
print(df.isnull().sum())

--- Dataset Shape ---
Total Rows: 1200, Total Columns: 13

--- Data Column Types ---
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       1200 non-null   int64  
 1   gender                    1200 non-null   str    
 2   daily_social_media_hours  1200 non-null   float64
 3   platform_usage            1200 non-null   str    
 4   sleep_hours               1200 non-null   float64
 5   screen_time_before_sleep  1200 non-null   float64
 6   academic_performance      1200 non-null   float64
 7   physical_activity         1200 non-null   float64
 8   social_interaction_level  1200 non-null   str    
 9   stress_level              1200 non-null   int64  
 10  anxiety_level             1200 non-null   int64  
 11  addiction_level           1200 non-null   int64  
 12  depression_label          1200 non-null   in

In [2]:
# Strip whitespaces from column names and string columns to avoid index errors
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()

# Check distribution of categorical behaviors
print("Platform Usage Counts:\n", df['platform_usage'].value_counts())
print("\nSocial Interaction Levels:\n", df['social_interaction_level'].value_counts())

Platform Usage Counts:
 platform_usage
Instagram    411
TikTok       398
Both         391
Name: count, dtype: int64

Social Interaction Levels:
 social_interaction_level
medium    416
low       415
high      369
Name: count, dtype: int64


C:\Users\VISHAL JADHAV\AppData\Local\Temp\ipykernel_18992\4003090840.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


In [3]:
# Let's use NumPy vectorization to create custom conditional categories

# A. Categorize 'daily_social_media_hours' into usage tiers using np.where
df['screen_time_tier'] = np.where(df['daily_social_media_hours'] > 6.0, 'High Use',
                         np.where(df['daily_social_media_hours'] > 3.0, 'Moderate Use', 'Low Use'))

# B. Create a Sleep Deprivation flag using np.where
# Medical guidance suggests teenagers need at least $8$ hours of sleep.
df['is_sleep_deprived'] = np.where(df['sleep_hours'] < 8.0, 1, 0)

# C. Calculate a combined Mental Distress Score using NumPy basic arithmetic
# Combining Stress, Anxiety, and Addiction levels (Scale of 3 to 30)
df['mental_distress_score'] = df['stress_level'] + df['anxiety_level'] + df['addiction_level']

print(df[['daily_social_media_hours', 'screen_time_tier', 'sleep_hours', 'is_sleep_deprived', 'mental_distress_score']].head())

   daily_social_media_hours screen_time_tier  sleep_hours  is_sleep_deprived  \
0                       7.9         High Use          7.4                  1   
1                       1.9          Low Use          8.0                  0   
2                       1.3          Low Use          7.6                  1   
3                       7.4         High Use          6.9                  1   
4                       4.7     Moderate Use          4.9                  1   

   mental_distress_score  
0                      5  
1                     19  
2                      8  
3                     17  
4                     10  


In [4]:
# Calculate statistical indices across the dataset using NumPy operations
mean_distress = np.mean(df['mental_distress_score'])
std_distress = np.std(df['mental_distress_score'])
median_academic = np.median(df['academic_performance'])

print(f"Population Average Mental Distress Score: {mean_distress:.2f} (±{std_distress:.2f})")
print(f"Median Academic Performance GPA: {median_academic:.2f}")

# Extract a conditional subset using NumPy masks
high_risk_mask = (df['screen_time_tier'] == 'High Use') & (df['is_sleep_deprived'] == 1)
high_risk_teens = df[high_risk_mask]
print(f"\nNumber of teenagers with High Screen Time AND Sleep Deprivation: {len(high_risk_teens)}")

Population Average Mental Distress Score: 16.65 (±5.04)
Median Academic Performance GPA: 2.99

Number of teenagers with High Screen Time AND Sleep Deprivation: 269


In [5]:
# Group by screen time tiers and gender to extract complex aggregation summaries
summary_table = df.groupby(['screen_time_tier', 'gender']).agg(
    avg_sleep=('sleep_hours', 'mean'),
    avg_stress=('stress_level', 'mean'),
    avg_anxiety=('anxiety_level', 'mean'),
    avg_gpa=('academic_performance', 'mean'),
    total_count=('age', 'count')
).reset_index()

print("--- Aggregated Summary Profile ---")
print(summary_table)

# Create a multi-dimensional Pivot Table comparing platform selection and interaction level
pivot_distress = df.pivot_table(
    values='mental_distress_score',
    index='platform_usage',
    columns='social_interaction_level',
    aggfunc='mean'
)
print("\n--- Pivot Table: Average Mental Distress by Platform vs Social Interaction ---")
print(pivot_distress)

--- Aggregated Summary Profile ---
  screen_time_tier  gender  avg_sleep  avg_stress  avg_anxiety   avg_gpa  \
0         High Use  female   6.498780    5.859756     5.439024  3.051098   
1         High Use    male   6.480814    5.674419     5.709302  2.995349   
2          Low Use  female   6.710778    5.299401     5.628743  3.010180   
3          Low Use    male   6.373714    5.720000     5.280000  3.002800   
4     Moderate Use  female   6.353937    5.220472     5.885827  2.961575   
5     Moderate Use    male   6.376119    5.171642     5.712687  2.956903   

   total_count  
0          164  
1          172  
2          167  
3          175  
4          254  
5          268  

--- Pivot Table: Average Mental Distress by Platform vs Social Interaction ---
social_interaction_level       high        low     medium
platform_usage                                           
Both                      17.281553  16.395833  16.138889
Instagram                 16.661765  16.516667  17.006452
T

In [6]:
# Save the aggregated summary table to a CSV file for reporting or visualization dashboards
summary_table.to_csv('teen_mental_health_summary.csv', index=False)
print("Data aggregated successfully and exported to 'teen_mental_health_summary.csv'!")

Data aggregated successfully and exported to 'teen_mental_health_summary.csv'!
